# RAG Evaluation Lab — Part 2: Vertex AI Embeddings Benchmark

This notebook **extends** [`rag_evaluation_lab.ipynb`](./rag_evaluation_lab.ipynb) — it does not replace it.

Part 1 taught the fundamentals with a **keyword retriever** and a **TF‑IDF retriever** on a
tiny synthetic corpus. Part 2 reuses that exact same corpus, ground truth, and metric
functions, and adds a third retriever: **Google's `text-embedding-004` model via Vertex AI**.

By the end you will have a **side-by-side benchmark** of three retrieval strategies —
keyword overlap, TF‑IDF, and embeddings — measured with **Recall@k**, **Precision@k**, and
**MRR** (Mean Reciprocal Rank).

> 🎯 **Goal:** See, with real numbers, why production RAG systems use embeddings instead of
> keyword or TF‑IDF search — and practice wiring up a real cloud embedding API.

## Requirements

- A GCP project with billing enabled and the Vertex AI API turned on
- Local auth: `gcloud auth application-default login`
- `pip install google-cloud-aiplatform`

See the README's **GCP Setup** section for the full walkthrough if you haven't used Vertex AI before.


In [ ]:
# If you haven't already, install the Vertex AI SDK.
# %pip install google-cloud-aiplatform


## 1. Configuration

Set your GCP project ID and region below. The rest of the notebook doesn't need edits.


In [ ]:
# --- Fill in your own GCP project ---
GCP_PROJECT_ID = "your-gcp-project-id"   # e.g. "ramirez-rag-lab"
GCP_LOCATION = "us-central1"
EMBEDDING_MODEL = "text-embedding-004"
K = 3  # matches the @3 metrics used throughout Part 1

print(f"Project: {GCP_PROJECT_ID} | Location: {GCP_LOCATION} | Model: {EMBEDDING_MODEL}")


In [ ]:
import vertexai
from vertexai.language_models import TextEmbeddingModel, TextEmbeddingInput

vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)
embedding_model = TextEmbeddingModel.from_pretrained(EMBEDDING_MODEL)

print("✅ Vertex AI initialized.")


## 2. Same Corpus, Same Ground Truth, Same Metrics

This is copied **verbatim** from Part 1 so the benchmark is apples-to-apples. If you change
anything here, the comparison against the beginner notebook's numbers is no longer valid.


In [ ]:
from typing import List, Dict
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4)

# Our tiny "corpus" of documents (knowledge base) -- identical to Part 1
documents = [
    "RAG systems combine retrieval and generation using large language models.",
    "TF-IDF is a classic method to turn documents into numerical vectors based on word frequency.",
    "Vector databases store embeddings and let you search by semantic similarity.",
    "Precision and recall are standard information retrieval metrics used to evaluate search quality.",
    "Evaluation of LLMs often includes answer correctness, grounding, and hallucination analysis."
]
doc_ids = list(range(len(documents)))

# Our tiny set of user questions (queries) -- identical to Part 1
queries = [
    "How do RAG systems work?",
    "What is TF-IDF in information retrieval?",
    "How do we evaluate search with precision and recall?",
]

# Ground-truth relevance -- identical to Part 1
ground_truth_relevant = {
    0: [0, 4],   # RAG question: docs 0 and 4 are relevant
    1: [1],      # TF-IDF question: doc 1 is relevant
    2: [3],      # precision/recall question: doc 3 is relevant
}

print("Number of documents:", len(documents))
print("Number of queries:", len(queries))


In [ ]:
def recall_at_k(retrieved: List[int], relevant: List[int], k: int) -> float:
    """Compute Recall@k for a single query."""
    top_k = retrieved[:k]
    if not relevant:
        return 0.0
    hits = sum(1 for d in top_k if d in relevant)
    return hits / len(relevant)


def precision_at_k(retrieved: List[int], relevant: List[int], k: int) -> float:
    """Compute Precision@k for a single query."""
    top_k = retrieved[:k]
    if not top_k:
        return 0.0
    hits = sum(1 for d in top_k if d in relevant)
    return hits / len(top_k)


def tokenize(text: str) -> List[str]:
    """Very simple tokenizer: lowercase + split on non-letters."""
    return re.findall(r"[a-zA-Z]+", text.lower())


def keyword_retrieve(query: str, docs: List[str]) -> List[int]:
    """Rank docs by how many query words they contain (descending)."""
    q_tokens = tokenize(query)
    doc_scores = []
    for doc_id, doc in enumerate(docs):
        d_tokens = tokenize(doc)
        overlap = len(set(q_tokens) & set(d_tokens))
        doc_scores.append((doc_id, overlap))
    ranked = sorted(doc_scores, key=lambda x: (-x[1], x[0]))
    return [doc_id for doc_id, score in ranked]


tfidf_vectorizer = TfidfVectorizer(stop_words="english")
doc_tfidf = tfidf_vectorizer.fit_transform(documents)

def tfidf_retrieve(query: str, top_k: int = 5) -> List[int]:
    """Use TF-IDF + cosine similarity to rank documents for a query."""
    q_vec = tfidf_vectorizer.transform([query])
    sims = cosine_similarity(q_vec, doc_tfidf)[0]
    ranked_indices = np.argsort(-sims)
    return ranked_indices[:top_k].tolist()

print("✅ Baseline retrievers (keyword, TF-IDF) ready -- same functions as Part 1.")


## 3. A New Metric: Mean Reciprocal Rank (MRR)

Part 1 covered Recall@k and Precision@k, which only look at the **top k** results as a set.
They don't care whether the relevant doc was ranked 1st or 3rd within that window.

**MRR** fixes that by rewarding retrievers that put the relevant document **higher**:

> "For a single query, take 1 / (rank of the first relevant document). Average that over all queries."

- If the first relevant doc is rank 1 → reciprocal rank = 1.0 (perfect)
- If it's rank 2 → reciprocal rank = 0.5
- If it's rank 3 → reciprocal rank = 0.33
- If no relevant doc appears at all → reciprocal rank = 0.0

MRR is a good complement to Recall/Precision because it's **rank-sensitive**: it distinguishes
a retriever that gets the right answer in position 1 from one that buries it in position 5.


In [ ]:
def reciprocal_rank(retrieved: List[int], relevant: List[int]) -> float:
    """Reciprocal rank of the first relevant doc in a ranked list.

    Returns 0.0 if no relevant doc appears in `retrieved` at all.
    """
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


# Quick sanity check
print("RR when relevant doc is rank 1:", reciprocal_rank([4, 1, 2], [4]))
print("RR when relevant doc is rank 3:", reciprocal_rank([1, 2, 4], [4]))
print("RR when relevant doc is missing:", reciprocal_rank([1, 2, 3], [4]))


## 4. Vertex AI Embeddings Retriever

`text-embedding-004` turns text into a dense vector that captures **meaning**, not just word
overlap. Unlike TF‑IDF, it can match a query to a relevant document even when they don't share
any exact words.

One detail that trips people up: Vertex AI's embedding API wants you to declare a **task type**
for each piece of text you embed:

- `RETRIEVAL_DOCUMENT` — for the documents going into your index
- `RETRIEVAL_QUERY` — for the user's query at search time

These are trained differently under the hood, so using the right one for each side of the
comparison measurably improves retrieval quality. This is the most common mistake when
first wiring up Vertex embeddings for RAG.


In [ ]:
def get_embeddings(texts: List[str], task_type: str) -> np.ndarray:
    """Embed a batch of texts with the given Vertex AI task type."""
    inputs = [TextEmbeddingInput(text=t, task_type=task_type) for t in texts]
    embeddings = embedding_model.get_embeddings(inputs)
    return np.array([e.values for e in embeddings])


# Embed the corpus once, up front, as RETRIEVAL_DOCUMENT vectors
doc_embeddings = get_embeddings(documents, task_type="RETRIEVAL_DOCUMENT")
print("Document embeddings shape:", doc_embeddings.shape)


def vertex_retrieve(query: str, top_k: int = 5) -> List[int]:
    """Use Vertex AI text-embedding-004 + cosine similarity to rank documents."""
    q_embedding = get_embeddings([query], task_type="RETRIEVAL_QUERY")
    sims = cosine_similarity(q_embedding, doc_embeddings)[0]
    ranked_indices = np.argsort(-sims)
    return ranked_indices[:top_k].tolist()


print("✅ Vertex AI retriever ready.")


## 5. Evaluate All Three Retrievers

Same evaluation loop, run once per retriever, so the comparison is apples-to-apples.


In [ ]:
def run_eval(retrieve_fn, name: str) -> pd.DataFrame:
    """Run a retriever over every query and score it with Recall@k, Precision@k, MRR."""
    rows = []
    for qi, q in enumerate(queries):
        ranked_docs = retrieve_fn(q)
        relevant = ground_truth_relevant[qi]
        rows.append({
            "retriever": name,
            "query_id": qi,
            "query": q,
            "ranked_docs": ranked_docs,
            f"recall@{K}": recall_at_k(ranked_docs, relevant, k=K),
            f"precision@{K}": precision_at_k(ranked_docs, relevant, k=K),
            "mrr": reciprocal_rank(ranked_docs, relevant),
        })
    return pd.DataFrame(rows)


results_keyword = run_eval(lambda q: keyword_retrieve(q, documents), "keyword")
results_tfidf = run_eval(tfidf_retrieve, "tfidf")
results_vertex = run_eval(vertex_retrieve, "vertex_text-embedding-004")

all_results = pd.concat([results_keyword, results_tfidf, results_vertex], ignore_index=True)
all_results[["retriever", "query_id", f"recall@{K}", f"precision@{K}", "mrr"]]


## 6. Summary Table

Mean score per retriever, across all three queries.


In [ ]:
summary = (
    all_results
    .groupby("retriever")[[f"recall@{K}", f"precision@{K}", "mrr"]]
    .mean()
    .round(3)
    .sort_values("mrr", ascending=False)
)
summary


## 7. Chart: Retriever Comparison


In [ ]:
metrics = [f"recall@{K}", f"precision@{K}", "mrr"]
retrievers = summary.index.tolist()

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 4))
for i, retriever in enumerate(retrievers):
    offset = (i - (len(retrievers) - 1) / 2) * width
    ax.bar(x + offset, summary.loc[retriever, metrics], width, label=retriever)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title(f"Keyword vs TF-IDF vs Vertex AI ({EMBEDDING_MODEL})")
ax.legend()

plt.tight_layout()
plt.savefig("retrieval_benchmark_vertex.png", dpi=150)
plt.show()


## 8. Error Analysis

For each query, which retrievers missed a relevant document entirely (Recall@k == 0)?
This is more informative than the averages above on a corpus this small.


In [ ]:
misses = all_results[all_results[f"recall@{K}"] == 0][["retriever", "query_id", "query", "ranked_docs"]]

if misses.empty:
    print("No full misses -- every retriever found at least one relevant doc per query.")
else:
    print(misses.to_string(index=False))


## 9. Takeaways & Next Steps

**Fill this in after running the notebook against your own GCP project:**

- Which retriever had the highest MRR? Was it the one you expected?
- Did embeddings help most on the query where keyword/TF‑IDF struggled?
- On a corpus this tiny, how much should you trust the averages vs. the per-query detail?

### Where to go next

- Swap in a larger, real-world corpus (your own docs) instead of the synthetic one.
- Try a different Vertex AI embedding model or dimensionality.
- Add latency and cost tracking per retriever call.
- Feed the retrieved docs into an LLM and re-run the Part 1 answer-overlap score end-to-end.

> 🧩 **Key idea:** the metrics didn't change between Part 1 and Part 2 — only the retriever
> did. That's the point: a good evaluation harness lets you swap retrieval strategies in and
> compare them fairly, which is exactly what you'd do before shipping a production RAG system.
